In [2]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

In [14]:
train_df = pd.read_csv('train_data_bulk_trial.csv')
test_df = pd.read_csv('test_data_bulk_trial.csv')

In [16]:
TEXT_COL = "posts"
LABEL_COLS = ["target_1", "target_2", "target_3", "target_4"]
DIM_NAMES = LABEL_COLS

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [17]:
train_df = train_df.copy()
test_df  = test_df.copy()

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)
test_df[TEXT_COL]  = test_df[TEXT_COL].astype(str)

y_train_full = train_df[LABEL_COLS].astype(int).values
y_test = test_df[LABEL_COLS].astype(int).values

tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED)

In [18]:
def compute_pos_weight(df: pd.DataFrame, label_cols):
    # pos_weight = N_neg / N_pos for each label dimension
    y = df[label_cols].astype(int).values
    pos = y.sum(axis=0)
    neg = y.shape[0] - pos
    # avoid division by zero
    pos = np.clip(pos, 1, None)
    pos_weight = (neg / pos).astype(np.float32)
    return torch.tensor(pos_weight)

pos_weight = compute_pos_weight(tr_df, LABEL_COLS)
pos_weight

tensor([2.5212, 1.8988, 1.3540, 1.7675])

In [19]:
def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[[TEXT_COL] + LABEL_COLS].reset_index(drop=True))

ds_train = to_hf_dataset(tr_df)
ds_val   = to_hf_dataset(va_df)
ds_test  = to_hf_dataset(test_df)

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

MODEL_NAME_C = "microsoft/deberta-v3-base"

tokenizer_c = AutoTokenizer.from_pretrained(MODEL_NAME_C)
data_collator_c = DataCollatorWithPadding(tokenizer=tokenizer_c)

MAX_LEN_C = 512   # since you want full posts

/home/nina/venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [20]:
def tokenize_batch_c(batch):
    return tokenizer_c(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN_C,
    )
ds_train_tok_c = ds_train.map(tokenize_batch_c, batched=True)
ds_val_tok_c   = ds_val.map(tokenize_batch_c, batched=True)
ds_test_tok_c  = ds_test.map(tokenize_batch_c, batched=True)

# Trainer expects "labels" field; for multi-label make it float tensor
def add_labels(batch):
    labels = np.stack([batch[c] for c in LABEL_COLS], axis=1).astype(np.float32)
    batch["labels"] = labels
    return batch

ds_train_tok_c = ds_train_tok_c.map(add_labels, batched=True)
ds_val_tok_c   = ds_val_tok_c.map(add_labels, batched=True)
ds_test_tok_c  = ds_test_tok_c.map(add_labels, batched=True)

cols_to_remove = [TEXT_COL] + LABEL_COLS

ds_train_tok_c = ds_train_tok_c.remove_columns(cols_to_remove)
ds_val_tok_c   = ds_val_tok_c.remove_columns(cols_to_remove)
ds_test_tok_c  = ds_test_tok_c.remove_columns(cols_to_remove)


Map:   0%|          | 0/71850 [00:00<?, ? examples/s]

Map:   0%|          | 0/17963 [00:00<?, ? examples/s]

Map:   0%|          | 0/22454 [00:00<?, ? examples/s]

Map:   0%|          | 0/71850 [00:00<?, ? examples/s]

Map:   0%|          | 0/17963 [00:00<?, ? examples/s]

Map:   0%|          | 0/22454 [00:00<?, ? examples/s]

In [21]:


data_collator = DataCollatorWithPadding(tokenizer=tokenizer_c)

In [22]:
def fulltype_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    correct = (y_true == y_pred).sum(axis=1)
    return {
        "full_type_acc_all4": float((correct == 4).mean()),
        "at_least_3": float((correct >= 3).mean()),
        "at_least_2": float((correct >= 2).mean()),
        "at_least_1": float((correct >= 1).mean()),
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    # per-dim metrics
    macro_f1s = []
    bal_accs = []
    for k in range(labels.shape[1]):
        macro_f1s.append(f1_score(labels[:, k], preds[:, k], average="macro"))
        bal_accs.append(balanced_accuracy_score(labels[:, k], preds[:, k]))

    out = {
        "macro_f1_mean": float(np.mean(macro_f1s)),
        "balanced_acc_mean": float(np.mean(bal_accs)),
    }
    out.update(fulltype_metrics(labels, preds))
    return out

In [23]:
class WeightedBCETrainer(Trainer):
    def __init__(self, pos_weight: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(model.device)  # float32 shape (B,4)
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(model.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [24]:
# MODEL_NAME = "roberta-base"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_C)

model_c = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_C,
    num_labels=4,
    problem_type="multi_label_classification",
).to(DEVICE)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_c)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
import optuna
import shutil
from transformers import TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification

In [27]:


def objective(trial):
    lr = trial.suggest_float("learning_rate", 5e-6, 5e-5, log=True)
    wd = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    grad_acc = trial.suggest_categorical("grad_accum", [1, 2, 4])

    # Keep each trial cheap
    args = TrainingArguments(
        output_dir=f"optuna_deberta/trial_{trial.number}",
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        logging_steps=200,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1_mean",
        greater_is_better=True,

        learning_rate=lr,
        weight_decay=wd,
        warmup_ratio=warmup,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=grad_acc,

        num_train_epochs=1,   # <-- cheap trials
        fp16=True,
        seed=SEED,
        report_to="none",
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        "microsoft/deberta-v3-base",
        num_labels=4,
        problem_type="multi_label_classification",
    ).to(DEVICE)

    trainer = WeightedBCETrainer(
        pos_weight=pos_weight,
        model=model,
        args=args,
        train_dataset=ds_train_tok_c,
        eval_dataset=ds_val_tok_c,
        tokenizer=tokenizer_c,
        data_collator=data_collator_c,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    metrics = trainer.evaluate(ds_val_tok_c)

    # cleanup to save disk
    shutil.rmtree(args.output_dir, ignore_errors=True)

    return metrics["eval_macro_f1_mean"]

In [28]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

study.best_value, study.best_params

[I 2026-03-06 16:06:28,370] A new study created in memory with name: no-name-19c09e58-f0cd-4709-9b34-858cc646be7a
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3153586/743699552.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss,Macro F1 Mean,Balanced Acc Mean,Full Type Acc All4,At Least 3,At Least 2,At Least 1
1000,0.876100,0.850715,0.601447,0.621440,0.169849,0.543339,0.849079,0.976841
2000,0.693300,0.703882,0.749249,0.743657,0.462005,0.742916,0.922674,0.989256
3000,0.619200,0.642197,0.785372,0.789040,0.561933,0.758893,0.910928,0.984691
4000,0.589100,0.576985,0.804486,0.810126,0.596393,0.780104,0.917775,0.985470
5000,0.507800,0.542731,0.811436,0.818152,0.611869,0.786840,0.920559,0.985915
6000,0.515400,0.605663,0.811093,0.819196,0.606914,0.783666,0.923454,0.986695
7000,0.580900,0.492267,0.810714,0.820128,0.599955,0.787118,0.925013,0.986250


[I 2026-03-06 16:33:30,113] Trial 0 finished with value: 0.8114363855703018 and parameters: {'learning_rate': 1.737488956156453e-05, 'weight_decay': 0.030119014625492747, 'warmup_ratio': 0.09958282110082584, 'grad_accum': 1}. Best is trial 0 with value: 0.8114363855703018.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3153586/743699552.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss,Macro F1 Mean,Balanced Acc Mean,Full Type Acc All4,At Least 3,At Least 2,At Least 1
1000,0.891500,0.892746,0.367084,0.522199,0.026889,0.260090,0.651172,0.907699
2000,0.881300,0.877627,0.537666,0.576232,0.130435,0.497356,0.843512,0.984301
3000,0.897200,0.898693,0.358866,0.500123,0.123866,0.397317,0.781161,0.983355
4000,0.912800,0.898814,0.324897,0.500058,0.056171,0.299449,0.675778,0.956856


[I 2026-03-06 16:49:32,564] Trial 1 finished with value: 0.537666473644003 and parameters: {'learning_rate': 2.854682654388482e-05, 'weight_decay': 0.06343009572773277, 'warmup_ratio': 0.14946487338201833, 'grad_accum': 1}. Best is trial 0 with value: 0.8114363855703018.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3153586/743699552.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


...

Step,Training Loss,Validation Loss,Macro F1 Mean,Balanced Acc Mean,Full Type Acc All4,At Least 3,At Least 2,At Least 1
1000,0.880100,0.887382,0.489388,0.527569,0.106218,0.460613,0.827701,0.977565
2000,0.878400,0.877378,0.509704,0.568629,0.110282,0.413962,0.792184,0.971720
3000,0.819500,0.832157,0.620955,0.626719,0.211602,0.549073,0.843957,0.977342
4000,0.779000,0.804888,0.647993,0.661353,0.252018,0.600512,0.854312,0.977398
5000,0.742700,0.740262,0.682456,0.696740,0.304125,0.644380,0.884819,0.979847
6000,0.689300,0.686022,0.743173,0.742851,0.441407,0.730947,0.913767,0.986584
7000,0.650700,0.621120,0.764913,0.770920,0.483995,0.745031,0.915883,0.986472
8000,0.606900,0.583796,0.783042,0.789146,0.524968,0.763124,0.923565,0.987196


[I 2026-03-07 01:29:57,434] Trial 13 finished with value: 0.7830418605419865 and parameters: {'learning_rate': 1.4768173701834954e-05, 'weight_decay': 0.030711679367179015, 'warmup_ratio': 0.012541378754793178, 'grad_accum': 2}. Best is trial 4 with value: 0.8373832853982659.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3153586/743699552.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss,Macro F1 Mean,Balanced Acc Mean,Full Type Acc All4,At Least 3,At Least 2,At Least 1
1000,0.868300,0.852346,0.602275,0.615803,0.178033,0.571842,0.878639,0.986083
2000,0.693800,0.649938,0.744877,0.758166,0.417247,0.724434,0.908089,0.984357
3000,0.571900,0.559441,0.789176,0.798993,0.550966,0.758170,0.918777,0.987586
4000,0.531000,0.546399,0.801161,0.803218,0.578912,0.782386,0.926349,0.989478
5000,0.507000,0.497727,0.803358,0.818166,0.581584,0.773200,0.919390,0.986194
6000,0.493200,0.490213,0.818312,0.824619,0.614096,0.796025,0.931025,0.989311
7000,0.492600,0.472573,0.822702,0.828487,0.620052,0.802928,0.935478,0.990202
8000,0.484200,0.469596,0.820069,0.830341,0.611758,0.797584,0.931303,0.989868


[I 2026-03-07 02:13:36,361] Trial 14 finished with value: 0.8227018966639694 and parameters: {'learning_rate': 6.59265368858069e-06, 'weight_decay': 0.04684797026608755, 'warmup_ratio': 0.1592428805557303, 'grad_accum': 2}. Best is trial 4 with value: 0.8373832853982659.


(0.8373832853982659,
 {'learning_rate': 6.686235955128919e-06,
  'weight_decay': 0.0769029210352457,
  'warmup_ratio': 0.1158557102349933,
  'grad_accum': 1})

In [29]:
best = study.best_params


In [30]:
best

{'learning_rate': 6.686235955128919e-06,
 'weight_decay': 0.0769029210352457,
 'warmup_ratio': 0.1158557102349933,
 'grad_accum': 1}

In [31]:
best = study.best_params

args_final = TrainingArguments(
    output_dir="outputs_deberta_optuna_best",
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1_mean",
    greater_is_better=True,

    learning_rate=best["learning_rate"],
    weight_decay=best["weight_decay"],
    warmup_ratio=best["warmup_ratio"],

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=best["grad_accum"],

    num_train_epochs=3,
    fp16=True,
    seed=SEED,
    report_to="none",
)

model_final = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=4,
    problem_type="multi_label_classification",
).to(DEVICE)

trainer_final = WeightedBCETrainer(
    pos_weight=pos_weight,
    model=model_final,
    args=args_final,
    train_dataset=ds_train_tok_c,
    eval_dataset=ds_val_tok_c,
    tokenizer=tokenizer_c,
    data_collator=data_collator_c,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

trainer_final.train()
test_metrics = trainer_final.evaluate(ds_test_tok_c)
test_metrics

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3153586/743699552.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss,Macro F1 Mean,Balanced Acc Mean,Full Type Acc All4,At Least 3,At Least 2,At Least 1
1000,0.906600,0.897496,0.441790,0.522798,0.150866,0.551745,0.888326,0.990926
2000,0.876300,0.872922,0.544854,0.572377,0.133608,0.533987,0.861103,0.984245
3000,0.813200,0.820475,0.641825,0.640054,0.250237,0.625564,0.896342,0.987808
4000,0.724500,0.717827,0.722348,0.728469,0.379948,0.708735,0.907309,0.988922
5000,0.622200,0.652159,0.767410,0.765269,0.508991,0.754273,0.919167,0.987753
6000,0.601700,0.604221,0.773206,0.782965,0.514279,0.739520,0.914380,0.986918
7000,0.617800,0.559687,0.779683,0.797676,0.533653,0.742304,0.905862,0.982687
8000,0.531600,0.559938,0.808736,0.805607,0.592663,0.795635,0.936759,0.991538
9000,0.499000,0.529174,0.809221,0.816170,0.597896,0.784835,0.925792,0.988532
10000,0.573900,0.514950,0.806379,0.820784,0.592329,0.776040,0.919223,0.985693


{'eval_loss': 0.5149348974227905,
 'eval_macro_f1_mean': 0.8394253434228716,
 'eval_balanced_acc_mean': 0.8422741649241743,
 'eval_full_type_acc_all4': 0.6597042843145987,
 'eval_at_least_3': 0.8216798788634542,
 'eval_at_least_2': 0.9422819987530061,
 'eval_at_least_1': 0.9918054689587601,
 'eval_runtime': 139.2473,
 'eval_samples_per_second': 161.253,
 'eval_steps_per_second': 20.158,
 'epoch': 1.336079719423259}

In [18]:
save_dir = "deberta_mbti_finetuned_optuna_best"

trainer_final.save_model(save_dir)
tokenizer_c.save_pretrained(save_dir)

('deberta_mbti_finetuned_optuna_best/tokenizer_config.json',
 'deberta_mbti_finetuned_optuna_best/special_tokens_map.json',
 'deberta_mbti_finetuned_optuna_best/spm.model',
 'deberta_mbti_finetuned_optuna_best/added_tokens.json',
 'deberta_mbti_finetuned_optuna_best/tokenizer.json')